In [382]:
import requests 
import locale
from bs4 import BeautifulSoup

import pandas as pd
import mwparserfromhell
import re
import datetime as dt

In [383]:
locale.setlocale(locale.LC_TIME, 'fr_FR.UTF-8') 

'fr_FR.UTF-8'

In [370]:
def is_float(element):
    try:
        if type(element) == str: 
            float(element)
            return True
        else:
            return False
    except ValueError:
        return False

# Get all stations url from the main url

In [10]:
MAIN_URL = "https://fr.wikipedia.org/wiki/Liste_des_stations_du_m%C3%A9tro_de_Paris"
TABLE_CLASS = "wikitable sortable center jquery-tablesorter"

In [11]:
page = requests.get(MAIN_URL)
soup = BeautifulSoup(page.content, 'html.parser')

In [32]:
for t in soup.find_all("table"): 
    for a in t.find_all('a', href=True, text=True):
        print(a.text, a["href"])

nom précédent /wiki/Liste_des_stations_du_m%C3%A9tro_parisien_ayant_chang%C3%A9_de_nom
Abbesses /wiki/Abbesses_(m%C3%A9tro_de_Paris)
Alésia /wiki/Al%C3%A9sia_(m%C3%A9tro_de_Paris)
Alexandre Dumas /wiki/Alexandre_Dumas_(m%C3%A9tro_de_Paris)
Alma - Marceau /wiki/Alma_-_Marceau_(m%C3%A9tro_de_Paris)
Anatole France /wiki/Anatole_France_(m%C3%A9tro_de_Paris)
Levallois-Perret /wiki/Levallois-Perret
Anvers /wiki/Anvers_(m%C3%A9tro_de_Paris)
Argentine /wiki/Argentine_(m%C3%A9tro_de_Paris)
Arts et Métiers /wiki/Arts_et_M%C3%A9tiers_(m%C3%A9tro_de_Paris)
Assemblée nationale /wiki/Assembl%C3%A9e_nationale_(m%C3%A9tro_de_Paris)
Aubervilliers - Pantin - Quatre Chemins /wiki/Aubervilliers_-_Pantin_-_Quatre_Chemins_(m%C3%A9tro_de_Paris)
Aubervilliers /wiki/Aubervilliers
Avenue Émile-Zola /wiki/Avenue_%C3%89mile-Zola_(m%C3%A9tro_de_Paris)
Avron /wiki/Avron_(m%C3%A9tro_de_Paris)
Balard /wiki/Balard_(m%C3%A9tro_de_Paris)
Barbès - Rochechouart /wiki/Barb%C3%A8s_-_Rochechouart_(m%C3%A9tro_de_Paris)
Basili

In [217]:
URL_RAW = 'https://fr.wikipedia.org/w/index.php?title={}&action=raw'

In [219]:
t = mwparserfromhell.parse(subpage.content)

In [403]:
remove_space = '^[ \t]+|[ \t]+$'
text_between_double_brackets = "\[([^[\]]*)\]"

infobox = {}

for t in soup.find_all("table"): 
    for a in t.find_all('a', href=True, text=True):
        SUBPAGE = a["href"]
        subpage = requests.get(URL_RAW.format(SUBPAGE.replace('/wiki/', '')))
        parsed_content = mwparserfromhell.parse(subpage.content)
        for template in parsed_content.filter_templates(): 
            if template.name == 'Infobox Station de métro\n':
                station_name = a['title']
                infobox[station_name] = {}
                infobox[station_name]['url'] = URL_RAW.format(SUBPAGE.replace('/wiki/', ''))
                params = template.params
                for param in params: 
                    param_name = re.sub(remove_space, '', str(param.name))
            
                    # convert to lists 
                    param_value = str(param.value) \
                                  .rstrip() \
                                  .replace('{', '[') \
                                  .replace('}', ']')
                    param_value = re.sub(remove_space, '', param_value)
                    if len(re.findall(text_between_double_brackets, param_value)):
                        param_value = re.findall(text_between_double_brackets, param_value)

                    # convert to float 
                    if is_float(param_value):
                        param_value = float(param_value)

                    # convert to date 
                    #if param_name == 'mise en service': 
                    #    param_value = [e.split('|')[1] for e in param_value]
                    #    param_value = ' '.join(param_value)
                    #    param_value = dt.datetime.strptime(param_value, '%d %B %Y')

                    infobox[station_name][param_name] = param_value

In [409]:
pd.DataFrame(infobox).T.to_csv('station_extract.csv')

# Sandbox

In [245]:
SUBPAGE = 'Bobigny_-_Pantin_-_Raymond_Queneau_(m%C3%A9tro_de_Paris)'
subpage = requests.get(URL_RAW.format(SUBPAGE))

# To do:

- Get coordinates with right format 
- Get dates with right format 
- Get edges with line and url to station 

Stations
- Name 
- Line 
    - Mise en service 
    - Connection
- Coordinates 